<a href="https://colab.research.google.com/github/marstonsward/AAI-521-Computer-Vision-Image-Classification-Project/blob/main/main_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Truth in Pixels: AI-Generated Image Detection

**Team**: Marston Ward, Jasper Dolar, Victor Salcedo  
**Course**: AAI-521 Computer Vision  
**Date**: November 2025

---

## 🎯 Project Overview

This notebook demonstrates a complete machine learning pipeline for detecting AI-generated images using deep learning. We compare three approaches:

1. **Custom CNN** - Baseline architecture built from scratch
2. **ResNet50** - Transfer learning with frozen pretrained backbone
3. **EfficientNet-B2** - Advanced transfer learning for high-resolution images

### Dataset
- **Source**: Hugging Face (`Hemg/AI-Generated-vs-Real-Images-Datasets`)
- **Classes**: Real (0) vs AI-Generated (1)
- **Split**: 80% train, 10% validation, 10% test

### Workflow
```
Data Preparation → Model Training → Evaluation → Comparison
```

---

## 1. Setup and Installation

Import required libraries and custom modules.

In [ ]:
# Google Colab Setup - Clone repo and install dependencies
import sys
import os
import shutil

# Check if running on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Setting up Google Colab environment...")
    
    # Define repo name
    repo_name = 'AAI-521-Computer-Vision-Image-Classification-Project'
    
    # Navigate to /content first to avoid nested directories
    os.chdir('/content')
    
    # Remove existing repo if it exists (handles corrupt clones)
    if os.path.exists(repo_name):
        print(f"🗑️  Removing existing {repo_name}...")
        shutil.rmtree(repo_name)
        print("✅ Cleaned up old repository")
    
    # Clone fresh repository
    !git clone https://github.com/marstonsward/AAI-521-Computer-Vision-Image-Classification-Project.git
    print("✅ Repository cloned")
    
    # Change to project directory
    os.chdir(repo_name)
    print(f"📂 Working directory: {os.getcwd()}")
    
    # Debug: List what's actually in the directory
    print(f"📋 Contents: {', '.join(os.listdir('.'))}")
    
    # Install dependencies
    %pip install -q datasets torch torchvision matplotlib seaborn scikit-learn
    print("✅ Dependencies installed")
    
    # Verify src directory exists
    if os.path.exists('src'):
        print("✅ src/ directory found")
        print(f"📦 src/ contains: {', '.join(os.listdir('src'))}")
    else:
        print("❌ ERROR: src/ directory not found!")
        print(f"❌ Available: {os.listdir('.')}")
else:
    print("💻 Running locally")

🔧 Setting up Google Colab environment...
🗑️  Removing existing AAI-521-Computer-Vision-Image-Classification-Project...
✅ Cleaned up old repository
Cloning into 'AAI-521-Computer-Vision-Image-Classification-Project'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (74/74), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 77 (delta 24), reused 35 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 9.65 MiB | 18.79 MiB/s, done.
Resolving deltas: 100% (24/24), done.
remote: Total 77 (delta 24), reused 35 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 9.65 MiB | 18.79 MiB/s, done.
Resolving deltas: 100% (24/24), done.
✅ Repository cloned
📂 Working directory: /content/AAI-521-Computer-Vision-Image-Classification-Project
✅ Repository cloned
📂 Working directory: /content/AAI-521-Compu

In [ ]:
# Standard library imports
import sys
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt

# Add project directory to Python path for module imports
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"📂 Added to path: {project_root}")

# Verify we can see the src package
if (project_root / 'src').exists():
    print(f"✅ src package accessible at: {project_root / 'src'}")
else:
    print(f"❌ WARNING: src directory not found at {project_root}")

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim

# Custom modules
from src.models import CustomCNN, get_resnet50, get_efficientnet_b2, count_parameters
from src.data import prepare_data, create_dataloaders
from src.training import Trainer, evaluate_model
from src.visualization import (plot_training_history, plot_confusion_matrix,
                               compare_models, print_classification_report)
from src.eda import plot_class_distribution, visualize_samples

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ All modules imported successfully")

## 2. Data Preparation

Load and split the dataset from Hugging Face.

In [ ]:
%%time
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

# Load and split dataset
print("\n📥 Loading dataset from Hugging Face...")
(train_images, train_labels,
 val_images, val_labels,
 test_images, test_labels) = prepare_data(seed=SEED)

print(f"\n📊 Dataset Split:")
print(f"   Training:   {len(train_images):,} images")
print(f"   Validation: {len(val_images):,} images")
print(f"   Test:       {len(test_images):,} images")

In [ ]:
# Create DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader, val_loader, test_loader = create_dataloaders(
    train_images, train_labels,
    val_images, val_labels,
    test_images, test_labels,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

print(f"✅ DataLoaders created:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

## 3. Exploratory Data Analysis

Visualize the dataset distribution and sample images.

In [ ]:
# Class distribution using EDA module
plot_class_distribution(
    train_labels,
    class_names=['Real', 'AI-Generated'],
    colors=['#3498db', '#e74c3c'],
    title='Training Set Class Distribution'
)

In [ ]:
# Sample images using EDA module
visualize_samples(
    train_images,
    train_labels,
    class_names=['Real', 'AI-Generated'],
    samples_per_class=4,
    colors=['#3498db', '#e74c3c']
)

## 4. Model 1: Custom CNN

Baseline convolutional neural network with 3 conv layers, batch normalization, and dropout.

In [ ]:
%%time
# Create model
cnn_model = CustomCNN(num_classes=1).to(device)
trainable, total = count_parameters(cnn_model)
print(f"📊 CNN Parameters: {trainable:,} trainable / {total:,} total")

# Setup training
criterion_cnn = nn.BCEWithLogitsLoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=0.001)

trainer_cnn = Trainer(cnn_model, device, criterion_cnn, optimizer_cnn, use_amp=True)

# Train
print("\n🚀 Training Custom CNN...")
history_cnn = trainer_cnn.fit(train_loader, val_loader, num_epochs=10,
                              save_path='models/cnn_best.pth')

In [ ]:
# Plot training history
plot_training_history(history_cnn, "Custom CNN Training History")

# Evaluate on test set (convert binary to 2-class for consistency)
# For CNN with BCEWithLogitsLoss, we need special handling
cnn_model.eval()
all_preds_cnn = []
all_labels_cnn = []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        preds = (torch.sigmoid(outputs) > 0.5).long().squeeze()
        all_preds_cnn.extend(preds.cpu().numpy())
        all_labels_cnn.extend(labels.cpu().numpy())

test_acc_cnn = np.mean(np.array(all_preds_cnn) == np.array(all_labels_cnn))
print(f"\n🎯 Test Accuracy: {test_acc_cnn:.4f}")

plot_confusion_matrix(all_labels_cnn, all_preds_cnn, title='CNN Confusion Matrix')
print_classification_report(all_labels_cnn, all_preds_cnn)

## 5. Model 2: ResNet50 Transfer Learning

Pretrained ResNet50 with frozen backbone and custom classifier.

In [ ]:
%%time
# Create model
resnet_model = get_resnet50(num_classes=2, freeze_backbone=True).to(device)
trainable, total = count_parameters(resnet_model)
print(f"📊 ResNet50 Parameters: {trainable:,} trainable / {total:,} total")

# Setup training
criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(resnet_model.fc.parameters(), lr=1e-4, weight_decay=1e-4)

trainer_resnet = Trainer(resnet_model, device, criterion_resnet, optimizer_resnet, use_amp=True)

# Train
print("\n🚀 Training ResNet50...")
history_resnet = trainer_resnet.fit(train_loader, val_loader, num_epochs=5,
                                    save_path='models/resnet50_best.pth')

In [ ]:
# Plot and evaluate
plot_training_history(history_resnet, "ResNet50 Training History")

test_acc_resnet, all_preds_resnet, all_labels_resnet = evaluate_model(resnet_model, test_loader, device)
print(f"\n🎯 Test Accuracy: {test_acc_resnet:.4f}")

plot_confusion_matrix(all_labels_resnet, all_preds_resnet, title='ResNet50 Confusion Matrix')
print_classification_report(all_labels_resnet, all_preds_resnet)

## 6. Model 3: EfficientNet-B2 Transfer Learning

Advanced architecture optimized for high-resolution images.

In [ ]:
%%time
# Create model
efficientnet_model = get_efficientnet_b2(num_classes=2, freeze_backbone=True).to(device)
trainable, total = count_parameters(efficientnet_model)
print(f"📊 EfficientNet-B2 Parameters: {trainable:,} trainable / {total:,} total")

# Setup training
criterion_eff = nn.CrossEntropyLoss()
optimizer_eff = optim.Adam(efficientnet_model.classifier[1].parameters(), lr=1e-4, weight_decay=1e-4)

trainer_eff = Trainer(efficientnet_model, device, criterion_eff, optimizer_eff, use_amp=True)

# Train
print("\n🚀 Training EfficientNet-B2...")
history_eff = trainer_eff.fit(train_loader, val_loader, num_epochs=5,
                              save_path='models/efficientnet_b2_best.pth')

In [ ]:
# Plot and evaluate
plot_training_history(history_eff, "EfficientNet-B2 Training History")

test_acc_eff, all_preds_eff, all_labels_eff = evaluate_model(efficientnet_model, test_loader, device)
print(f"\n🎯 Test Accuracy: {test_acc_eff:.4f}")

plot_confusion_matrix(all_labels_eff, all_preds_eff, title='EfficientNet-B2 Confusion Matrix')
print_classification_report(all_labels_eff, all_preds_eff)

## 7. Model Comparison and Results

Compare all three models side-by-side.

In [ ]:
# Compare test accuracies
results = {
    'Custom CNN': test_acc_cnn,
    'ResNet50': test_acc_resnet,
    'EfficientNet-B2': test_acc_eff
}

compare_models(results, metric='Test Accuracy')

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
for model_name, accuracy in results.items():
    print(f"{model_name:20s}: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("="*60)

# Identify best model
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]
print(f"\n🏆 Best Model: {best_model_name} with {best_accuracy:.4f} accuracy")

## 8. Conclusion and Next Steps

### Key Findings

1. **Custom CNN**: Baseline performance with lightweight architecture
2. **ResNet50**: Strong transfer learning performance with minimal training
3. **EfficientNet-B2**: Best for capturing fine-grained details in high-resolution images

### Model Selection Criteria

- **Speed**: Custom CNN (smallest, fastest inference)
- **Accuracy**: EfficientNet-B2 (typically best for this task)
- **Balance**: ResNet50 (good accuracy, reasonable speed)

### Future Improvements

1. **Data Augmentation**: Add more sophisticated augmentation techniques
2. **Fine-tuning**: Unfreeze backbone layers for transfer learning models
3. **Ensemble**: Combine predictions from multiple models
4. **Attention Mechanisms**: Add attention layers to focus on synthetic artifacts
5. **Larger Dataset**: Train on more diverse AI-generated images

### Deployment Recommendations

- **Production**: Use EfficientNet-B2 for best accuracy
- **Edge Devices**: Use Custom CNN for resource-constrained environments
- **Cloud API**: Use ResNet50 for balanced performance

---

**Project Complete!** All models trained, evaluated, and compared successfully. 🎉